# Kokoro TTS Server — Kaggle T4 GPU
**NovAI röstserver** — Kokoro 82M multilingual, gratis T4 GPU  
Kvalitet: 8.5/10 | Latens: ~20-50ms/mening på T4  

**Steg:**
1. Installera paket
2. Starta FastAPI-server
3. Starta localhost.run-tunnel
4. Kopiera URL → lägg i NovAI .env som `KOKORO_SERVER_URL`

In [ ]:
# Cell 1: Installera paket
import subprocess, sys

# System-deps (espeak-ng för fonemisering)
subprocess.run(['apt-get', 'install', '-y', '-q', 'espeak-ng', 'libsndfile1'], check=True)

# Python-paket
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kokoro>=0.9.4',
    'scipy>=1.10.0',
    'numpy>=1.24.0',
    'fastapi>=0.100.0',
    'uvicorn[standard]>=0.23.0',
    'soundfile>=0.12.0',
], check=True)

print('✓ Alla paket installerade')

In [ ]:
# Cell 2: Skriv server.py till disk
SERVER_CODE = '''
import os, re, logging, time, asyncio
from math import gcd
import numpy as np
import scipy.signal
from fastapi import FastAPI, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("kokoro")
app = FastAPI(title="Kokoro TTS")

VOICE = os.getenv("KOKORO_VOICE", "af_nova")
LANG  = os.getenv("KOKORO_LANG",  "z")
SPEED = float(os.getenv("KOKORO_SPEED", "1.0"))
_pipeline = None
_sem = asyncio.Semaphore(1)

_EXP_LUT = np.array([
    0,0,1,1,2,2,2,2,3,3,3,3,3,3,3,3,
    4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,
    5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,
    5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,
    6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,
    6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,
    6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,
    6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,
    7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,
    7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,
    7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,
    7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,
    7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,
    7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,
    7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,
    7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,
], dtype=np.int32)
_MULAW_CLIP = 32635
_MULAW_BIAS_14 = 33

def _lin2ulaw(pcm16):
    s = pcm16.astype(np.int32) >> 2
    mask = np.where(s >= 0, np.int32(0xFF), np.int32(0x7F))
    s_abs = np.minimum(np.abs(s), _MULAW_CLIP) + _MULAW_BIAS_14
    exp = _EXP_LUT[(s_abs >> 7) & 0xFF]
    mant = (s_abs >> (exp + 3)) & 0x0F
    return (((exp << 4) | mant) ^ mask).astype(np.uint8)

def _wav_to_mulaw_8k(wav, sr):
    if len(wav) == 0:
        raise ValueError("Tom audio")
    if sr != 8000:
        g = gcd(8000, sr)
        wav = scipy.signal.resample_poly(wav, 8000 // g, sr // g)
    pre = 0.85
    wav = np.concatenate([[wav[0]], wav[1:] - pre * wav[:-1]])
    rms = np.sqrt(np.mean(wav ** 2))
    if rms > 1e-6:
        wav = wav * min(0.126 / rms, 10.0)
    wav = np.clip(wav, -0.98, 0.98)
    return _lin2ulaw((np.clip(wav, -1.0, 1.0) * 32767).astype(np.int16)).tobytes()

def _normalize_text(text):
    text = re.sub(r"(\\d{2,3})-(\\d{2,3})", r"\\1 \\2", text)
    text = re.sub(r"(\\d[\\d\\s]*)\\s*kr\\b", r"\\1 kronor", text, flags=re.IGNORECASE)
    text = re.sub(r"(\\d[\\d\\s]*)\\s*%", r"\\1 procent", text)
    for k, v in {"osv.": "och så vidare", "etc.": "et cetera", "ca.": "cirka",
                 "bl.a.": "bland annat", "dvs.": "det vill säga"}.items():
        text = text.replace(k, v)
    return text

def _get_pipeline():
    global _pipeline
    if _pipeline is None:
        from kokoro import KPipeline
        log.info(f"Laddar Kokoro (lang={LANG}, voice={VOICE})...")
        _pipeline = KPipeline(lang_code=LANG)
        log.info("Kokoro redo ✓")
    return _pipeline

class TTSRequest(BaseModel):
    text: str
    voice: str = None
    speed: float = None

@app.get("/health")
def health():
    try:
        import torch
        cuda = torch.cuda.is_available()
    except ImportError:
        cuda = False
    return {"status": "ok", "voice": VOICE, "lang": LANG, "cuda": cuda}

@app.post("/tts")
async def synthesize(req: TTSRequest):
    text = _normalize_text(req.text.strip())
    if not text:
        raise HTTPException(400, "text saknas")
    if len(text) > 500:
        raise HTTPException(400, "max 500 tecken")
    voice = req.voice or VOICE
    speed = req.speed or SPEED
    t0 = time.time()
    pipeline = _get_pipeline()
    async with _sem:
        loop = asyncio.get_running_loop()
        def _run():
            chunks = []
            for _, _, audio in pipeline(text, voice=voice, speed=speed):
                if audio is not None and len(audio) > 0:
                    chunks.append(np.array(audio, dtype=np.float32))
            return np.concatenate(chunks) if chunks else np.array([], dtype=np.float32)
        wav = await loop.run_in_executor(None, _run)
    if len(wav) == 0:
        raise HTTPException(500, "Tom audio — kontrollera voice/lang")
    mulaw = _wav_to_mulaw_8k(wav, 24000)
    ms = int((time.time() - t0) * 1000)
    log.info(f"TTS {ms}ms | {len(text)} tecken | {len(mulaw)}B")
    return Response(content=mulaw, media_type="audio/basic")

@app.on_event("startup")
async def startup():
    try:
        pipeline = _get_pipeline()
        list(pipeline("Hej välkommen.", voice=VOICE, speed=1.0))
        log.info("Server varm ✓")
    except Exception as e:
        log.warning(f"Warmup misslyckades: {e}")
'''

with open('/kaggle/working/kokoro_server.py', 'w') as f:
    f.write(SERVER_CODE)

print('✓ kokoro_server.py skrivet')

In [ ]:
# Cell 3: Starta servern i bakgrunden
import subprocess, time, os

env = os.environ.copy()
env['KOKORO_VOICE'] = 'af_nova'  # Byt röst här om du vill
env['KOKORO_LANG']  = 'z'       # z = multilingual (bäst för svenska)
env['PORT']         = '8000'

proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'kokoro_server:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/kaggle/working',
    stdout=open('/kaggle/working/server.log', 'w'),
    stderr=subprocess.STDOUT,
    start_new_session=True,
    env=env,
)

print(f'Server startad (PID {proc.pid}) — väntar på warmup...')
time.sleep(30)  # Kokoro laddar modellen (~20-30s första gången

# Visa loggen
with open('/kaggle/working/server.log') as f:
    print(f.read()[-3000:])

In [ ]:
# Cell 4: Testa lokalt
import urllib.request, json

try:
    # Health check
    with urllib.request.urlopen('http://localhost:8000/health', timeout=10) as r:
        health = json.loads(r.read())
    print('Health:', health)

    # TTS test
    req = json.dumps({'text': 'Hej, välkommen till NovAI!'}).encode()
    with urllib.request.urlopen(
        urllib.request.Request('http://localhost:8000/tts',
                               data=req, headers={'Content-Type': 'application/json'}),
        timeout=15
    ) as r:
        audio = r.read()
    print(f'✓ TTS fungerar — {len(audio)} bytes mulaw')
    
    # Spara som WAV för att lyssna
    import numpy as np, scipy.io.wavfile
    # Konvertera mulaw → pcm16 för WAV-fil
    mu = np.frombuffer(audio, dtype=np.uint8).astype(np.int32)
    mu = mu ^ 0xFF
    exp = (mu >> 4) & 0x07
    mant = mu & 0x0F
    t = ((mant << 1) | 1) << (exp + 2)
    sign_mask = np.where((mu ^ 0xFF) & 0x80, 1, -1)  
    pcm16 = (sign_mask * (t - 33)).astype(np.int16) * 4
    scipy.io.wavfile.write('/kaggle/working/test_kokoro.wav', 8000, pcm16)
    print('✓ Sparat test_kokoro.wav — lyssna under Output-fliken!')

except Exception as e:
    print(f'FEL: {e}')
    with open('/kaggle/working/server.log') as f:
        print(f.read()[-2000:])

In [ ]:
# Cell 5: Starta localhost.run-tunnel → ger publik URL
# KOPIERA URL-en och lägg i NovAI .env: KOKORO_SERVER_URL=https://xxxx.lhr.life
import subprocess, time

tunnel = subprocess.Popen(
    ['ssh', '-o', 'StrictHostKeyChecking=no', '-R', '80:localhost:8000', 'localhost.run'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print('Väntar på tunnel-URL...')
url = None
for _ in range(30):
    time.sleep(2)
    line = tunnel.stdout.readline().decode('utf-8', errors='replace')
    if line:
        print(line.strip())
    if 'lhr.life' in line or 'localhost.run' in line:
        import re
        m = re.search(r'https?://[\w.-]+\.lhr\.life', line)
        if m:
            url = m.group(0)
            break

if url:
    print(f'\n✅ TUNNEL KLAR!')
    print(f'\nKopiera till NovAI .env:')
    print(f'  TTS_PROVIDER=kokoro')
    print(f'  KOKORO_SERVER_URL={url}')
    print(f'  KOKORO_VOICE=af_nova')
    print(f'\nURL håller så länge Kaggle-sessionen lever (max 12h)')
else:
    print('Kunde inte hitta URL — kör cellen igen eller kolla loggen ovan')

In [ ]:
# Cell 6: Keepalive (håll sessionen vid liv)
# Kör denna cell om du vill att sessionen ska leva länge
import time

print('Keepalive aktiv — avbryt med Stop-knappen')
i = 0
while True:
    time.sleep(60)
    i += 1
    # Hälsokoll varje minut
    try:
        import urllib.request
        urllib.request.urlopen('http://localhost:8000/health', timeout=5)
        print(f'[{i}min] Server OK')
    except Exception as e:
        print(f'[{i}min] Server nere: {e}')